In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, confusion_matrix

from model import DERMANet


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


test_dataset = datasets.ImageFolder("dataset/test", transform)

test_loader = DataLoader(test_dataset, batch_size=16)


model = DERMANet(len(test_dataset.classes))

model.load_state_dict(torch.load("outputs/dermanet_best.pth"))

model.to(DEVICE)

model.eval()


y_true = []
y_pred = []


with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)

        outputs = model(images)

        _, preds = torch.max(outputs,1)

        y_true.extend(labels.numpy())

        y_pred.extend(preds.cpu().numpy())


print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=test_dataset.classes))


print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
